# Verify a published VoltageGPU attestation bundle, offline

This notebook downloads an evidence bundle that VoltageGPU publishes on its website, checks its
checksums, and verifies it **on this machine, without network access during verification**, with
[`voltage-verify`](https://github.com/VoltageGPU/voltage-verify) 0.2.3. Then it explains each line of
the verifier's output, breaks the bundle on purpose to show the checks fail, and ends with what the
result does **not** prove.

The bundle was captured on 9 October 2026 inside a VoltageGPU Confidential VM (SKU `rtx6000b-small`:
one NVIDIA RTX PRO 6000 Blackwell GPU in an Intel TDX guest), rented through the public API by the
weekly proof run. Its folder also holds the output printed inside the VM and the `nvidia-smi` and
kernel details: <https://voltagegpu.com/blog/two-proofs/evidence/rtx6000b-2026-10-09/README.txt>.

No account is needed. Requirements: Python 3.10+, `pip install "voltage-verify==0.2.3"`.

In [1]:
import hashlib, json, pathlib, subprocess, sys, urllib.request

EVIDENCE = "https://voltagegpu.com/blog/two-proofs/evidence/rtx6000b-2026-10-09"
FILES = ["bundle.json", "manifest.json", "SHA256SUMS"]
dl = pathlib.Path("downloads"); dl.mkdir(exist_ok=True)

for name in FILES:
    with urllib.request.urlopen(f"{EVIDENCE}/{name}", timeout=60) as r:
        (dl / name).write_bytes(r.read())
    print(f"{name:14} {(dl / name).stat().st_size:>8} bytes")

bundle.json      206984 bytes
manifest.json       269 bytes


SHA256SUMS          392 bytes


## 1. Checksums

`SHA256SUMS` was computed inside the VM, right after the bundle was written. Recomputing the hashes
here shows the files were served byte for byte. (It does not prove anything about their content yet:
anyone can publish a file and its hash. The signatures checked below do that.)

In [2]:
published = {line.split()[1]: line.split()[0]
             for line in (dl / "SHA256SUMS").read_text().splitlines() if line.strip()}
for name in ("manifest.json", "bundle.json"):
    digest = hashlib.sha256((dl / name).read_bytes()).hexdigest()
    print(f"{name:14} {'OK' if digest == published[name] else 'MISMATCH'}  {digest}")

manifest.json  OK  1d24e25c7623367d3d71082fae1f22e4f755a3c39fd9e82ef491b6e14beb506b
bundle.json    OK  cbe67c5fc88cf618e7dcfef6eaf31c3a23ee92a4b47e923f2f63e3f19ecc106b


## 2. The manifest and its challenge

The manifest is the small JSON document both hardware proofs are bound to. It carries a random
`challenge`; for a bundle you produce yourself you draw it on your own machine and pass it to the
VM, so an old bundle cannot be replayed to you. Here the challenge was drawn by the publisher's
script, so we read it from the manifest and pass it to the verifier with `--challenge`.

In [3]:
manifest = json.loads((dl / "manifest.json").read_text())
print(json.dumps(manifest, indent=2))
CHALLENGE = manifest["challenge"]

{
  "challenge": "627ba9c51d4719038e917c7431675b183ac576c819d712178dc8a100f4409a45",
  "created_at": "2026-10-09T21:53:58+00:00",
  "format": "voltage-verify-manifest/1",
  "statement": null,
  "workload": {
    "artifacts": [],
    "extra": {},
    "image": null
  }
}


## 3. Verify, offline

`voltage-verify verify` runs every check and prints one line per check. Options used:

* `--offline`: read the Intel collateral (TCB info, QE identity, CRLs) embedded in the bundle, itself
  checked against the pinned Intel SGX Root CA, instead of fetching it from Intel. NVIDIA's signing
  keys always come from the bundle's JWKS, trusted only through their certificate chain to a pinned
  NVIDIA NRAS intermediate.
* `--challenge`: the challenge the bundle must answer.
* `--hwmodel GB20X --gpus 1`: the GPU model and count NVIDIA's token must report (RTX PRO 6000
  Blackwell is `GB20X`).

Exit code 0 means verified, 1 means a required check failed, 2 means the bundle or arguments are
unusable.

In [4]:
def verify(*args):
    cmd = [sys.executable, "-m", "voltage_verify", "verify", *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    shown = [CHALLENGE[:8] + "..." if a == CHALLENGE else a for a in cmd[2:]]
    print("$ python -m", " ".join(shown))
    print(r.stdout + r.stderr)
    print("exit code:", r.returncode)
    return r

print(subprocess.run([sys.executable, "-m", "voltage_verify", "--version"],
                     capture_output=True, text=True).stdout)
ok = verify(str(dl / "bundle.json"), "--offline", "--challenge", CHALLENGE, "--hwmodel", "GB20X", "--gpus", "1")
assert ok.returncode == 0 and "RESULT: VERIFIED" in ok.stdout

voltage-verify 0.2.3



$ python -m voltage_verify verify downloads\bundle.json --offline --challenge 627ba9c5... --hwmodel GB20X --gpus 1
[PASS] manifest.format       manifest is well formed
[PASS] manifest.commitments  bundle commitments match the manifest
[PASS] manifest.challenge    challenge is the one this verifier issued
[PASS] tdx.structure         TDX quote v4, TEE 0x81, 4940 bytes
[PASS] tdx.debug             TD attributes 0000001000000000 (0x10000000), DEBUG (bit 0) off: not a debug TD
[PASS] tdx.report_data       quote report_data equals SHA-512 of the manifest
[PASS] tdx.signatures        attestation key, QE binding, QE report and PCK chain verify up to the pinned Intel SGX Root CA
[PASS] tdx.collateral        TCB info and QE identity signed by Intel (bundle (fetched 2026-10-09T21:54:01+00:00))
[PASS] tdx.tcb               platform TCB UpToDate, QE UpToDate, TDX_03 UpToDate
[PASS] tdx.revocation        PCK certificate not revoked (57 entries in CRL)
[PASS] tdx.revocation        PCK CA certificate

## 4. What each line means

The reading below is the one from the tool's own documentation,
[`docs/WHAT_IT_PROVES.md`](https://github.com/VoltageGPU/voltage-verify/blob/main/docs/WHAT_IT_PROVES.md),
reproduced for the checks printed above and nothing more.

### Manifest

| Check | Passes when | Proves | Does not prove |
|---|---|---|---|
| `manifest.format` | the manifest has the expected shape | the bundle is parseable | anything |
| `manifest.commitments` | SHA-256 and SHA-512 of the canonical manifest equal the stored values | bookkeeping is consistent | anything: an attacker can recompute them |
| `manifest.challenge` | the challenge equals the one you passed with `--challenge` | the bundle was made for this session | anything without the TDX and NVIDIA checks below |

### Intel TDX

| Check | Passes when | Proves | Does not prove |
|---|---|---|---|
| `tdx.structure` | version 4 quote, TEE type TDX, ECDSA-P256 key, QE report and PCK chain present | it is a TDX quote | anything about its origin |
| `tdx.debug` | bit 0 (DEBUG) of the TD attributes is 0; with `--allow-debug` a set bit is a warning, not a failure | the TD does not run in debug mode, in which the host can read and modify its memory and CPU state | anything on its own: the TD attributes are only Intel's word once `tdx.signatures` passes |
| `tdx.report_data` | the quote's 64-byte `report_data` equals `SHA-512(manifest)` | the quote was requested with your manifest hash, so after your challenge | who requested it |
| `tdx.signatures` | attestation key signs header+report; QE report binds the key; PCK certificate signs the QE report; PCK chain verifies to the pinned Intel SGX Root CA and is within validity | the quote was produced by a genuine Intel TDX Quoting Enclave on Intel-provisioned hardware | that the platform's firmware is current |
| `tdx.collateral` | TCB info and QE identity carry Intel's signature and chain to the root | the collateral is Intel's | freshness (see `tdx.collateral_fresh`) |
| `tdx.tcb` | the platform's SVNs (from the PCK certificate and the quote's `tee_tcb_svn`) reach a level Intel rates `UpToDate` or `SWHardeningNeeded`; the TDX module identity matches; the Quoting Enclave identity matches and is `UpToDate` | the platform is patched to a level Intel accepts | that no unknown vulnerability exists |
| `tdx.revocation` | the PCK certificate and its CA are absent from Intel's CRLs | Intel has not revoked this platform's provisioning identity | anything if CRLs are missing (then the line says NOT checked) |

The quote's `MRTD` and `RTMR0..3` are printed as facts. They measure the VM's initial state
and boot chain. This tool does not compare them to a reference: whether a given `MRTD` is the
image you expect is a policy decision that belongs to you.

### NVIDIA

| Check | Passes when | Proves | Does not prove |
|---|---|---|---|
| `nvidia.signatures` | the verifier token and every device token verify (ES384/ES256 on the matching curve) with a key from the bundle's JWKS, listed once, whose x5c leaf names NVIDIA Corporation, holds that key, is issued by a pinned NRAS intermediate, GPU Intermediate 004 or 005 (or one NVIDIA's live JWKS presents) and was valid at the token's `iat`; issuer is NRAS | NVIDIA issued these claims | when, until `nvidia.freshness` |
| `nvidia.nonce` | `eat_nonce` equals `SHA-256(manifest)` | the claims were issued for your manifest hash, after your challenge | which VM: the TDX side does that |
| `nvidia.policy` | overall result true; every GPU: `measres` success, `secboot` true, `dbgstat` disabled, nonce matched, report signature verified, certificate chain validated; optional `--hwmodel` and `--gpus` | NVIDIA's service accepted the GPU's measurements and configuration | that the GPU executed your workload |
| `nvidia.freshness` | the verifier token's `iat` is within 15 minutes of the bundle's `created_at` | the token belongs to this attestation session | anything if the VM clock is wrong (the check then fails, which is the safe direction) |

The two summary lines (`TDX: MRTD ...` and `GPU: GPU-0 GB20X driver ...`) are facts read from the
quote and from NVIDIA's token, printed for you to compare against your own policy.

## 5. Break it on purpose

A replayed or foreign bundle answers a challenge you did not issue: `manifest.challenge` fails.

In [5]:
bad = verify(str(dl / "bundle.json"), "--offline", "--challenge", "0" * 64, "--hwmodel", "GB20X", "--gpus", "1")
assert bad.returncode != 0 and "NOT VERIFIED" in bad.stdout

$ python -m voltage_verify verify downloads\bundle.json --offline --challenge 0000000000000000000000000000000000000000000000000000000000000000 --hwmodel GB20X --gpus 1
[PASS] manifest.format       manifest is well formed
[PASS] manifest.commitments  bundle commitments match the manifest
[FAIL] manifest.challenge    challenge differs from the one this verifier issued (replay or foreign bundle)
[PASS] tdx.structure         TDX quote v4, TEE 0x81, 4940 bytes
[PASS] tdx.debug             TD attributes 0000001000000000 (0x10000000), DEBUG (bit 0) off: not a debug TD
[PASS] tdx.report_data       quote report_data equals SHA-512 of the manifest
[PASS] tdx.signatures        attestation key, QE binding, QE report and PCK chain verify up to the pinned Intel SGX Root CA
[PASS] tdx.collateral        TCB info and QE identity signed by Intel (bundle (fetched 2026-10-09T21:54:01+00:00))
[PASS] tdx.tcb               platform TCB UpToDate, QE UpToDate, TDX_03 UpToDate
[PASS] tdx.revocation        PCK c

One changed character inside the signed part of the Intel quote: the attestation key's signature no
longer verifies, so `tdx.signatures` fails, whatever the challenge.

In [6]:
bundle = json.loads((dl / "bundle.json").read_text())
quote = bundle["tdx"]["quote_b64"]
bundle["tdx"]["quote_b64"] = quote[:300] + ("A" if quote[300] != "A" else "B") + quote[301:]
(dl / "tampered.json").write_text(json.dumps(bundle))
tampered = verify(str(dl / "tampered.json"), "--offline", "--challenge", CHALLENGE, "--hwmodel", "GB20X", "--gpus", "1")
assert tampered.returncode != 0 and "NOT VERIFIED" in tampered.stdout

$ python -m voltage_verify verify downloads\tampered.json --offline --challenge 627ba9c5... --hwmodel GB20X --gpus 1
[PASS] manifest.format       manifest is well formed
[PASS] manifest.commitments  bundle commitments match the manifest
[PASS] manifest.challenge    challenge is the one this verifier issued
[PASS] tdx.structure         TDX quote v4, TEE 0x81, 4940 bytes
[PASS] tdx.debug             TD attributes 0000001000000000 (0x10000000), DEBUG (bit 0) off: not a debug TD
[PASS] tdx.report_data       quote report_data equals SHA-512 of the manifest
[FAIL] tdx.signatures        quote signature does not verify with the attestation key: 
[PASS] nvidia.signatures     2 NVIDIA-signed tokens verify, keys from the bundle chained to a pinned NRAS intermediate (offline)
[PASS] nvidia.nonce          NRAS eat_nonce equals SHA-256 of the manifest
[PASS] nvidia.policy         overall result true, every GPU: measurements ok, secure boot on, debug off, nonce matched, report signature and chain val

## 6. The machine-readable report

`--json` prints the same checks plus the facts the verifier read: TD attributes, `MRTD`, `RTMR0..3`,
TCB status, the GPU claims. These are the values a policy of yours would compare against a reference.

In [7]:
r = subprocess.run([sys.executable, "-m", "voltage_verify", "verify", str(dl / "bundle.json"), "--offline",
                    "--challenge", CHALLENGE, "--hwmodel", "GB20X", "--gpus", "1", "--json"],
                   capture_output=True, text=True)
report = json.loads(r.stdout)
print("ok:", report["ok"], "| checks:", len(report["checks"]), "| facts:", sorted(report["facts"]))
tdx = report["facts"]["tdx"]
for key in ("mr_td", "mr_seam", "td_attributes", "xfam", "tee_tcb_svn", "fmspc", "tcb_status", "tcb_level_date", "qe_status"):
    print(f"tdx.{key:16} {tdx[key]}")
for i, rtmr in enumerate(tdx["rtmr"]):
    print(f"tdx.rtmr{i:<12} {rtmr}")
print("tdx.tdx_module     ", tdx["tdx_module"])
nvidia = report["facts"]["nvidia"]
print("nvidia.mode        ", nvidia["mode"], "| issued_at", nvidia["issued_at"], "| expires_at", nvidia["expires_at"])
for device in nvidia["devices"]:
    print("nvidia.device      ", json.dumps(device))

ok: True | checks: 15 | facts: ['artifacts', 'commitment_sha256', 'commitment_sha512', 'nvidia', 'tdx']
tdx.mr_td            eea8b6a814569a52bd1e12f6b869bb2d9c0c8a7a43e658ffc3b42c199f116157ea7f04d359c7fdfd8ac483152cc13542
tdx.mr_seam          476a2997c62bccc78370913d0a80b956e3721b24272bc66c4d6307ced4be2865c40e26afac75f12df3425b03eb59ea7c
tdx.td_attributes    0000001000000000
tdx.xfam             e702060000000000
tdx.tee_tcb_svn      05030400000000000000000000000000
tdx.fmspc            20a06d080000
tdx.tcb_status       UpToDate
tdx.tcb_level_date   2025-08-13T00:00:00Z
tdx.qe_status        UpToDate
tdx.rtmr0            f9ac3522e05572c2ede103eb9f243bebddee62d8e8db65a526736ea2bd5eb6dbe69a5ff8082c8050865b5ddadae4fa8a
tdx.rtmr1            965b54adf526d37237906189a31d9b058c39fac8909d9689e3340505aac4176deed2c2f3374b226304b23137d4832ae9
tdx.rtmr2            d9d5a5c17828c2b8b390a7df2fc5f060211342001b8044705182c400e4ab24c761630531ff15f2b0defd384c08816f3a
tdx.rtmr3            0000000000000000000

## 7. What VERIFIED does not mean

From the tool's README and `docs/WHAT_IT_PROVES.md`, unchanged:

Verified does **not** mean:

* that the GPU *executed* the image or model named in the manifest. Binding a manifest to a
  quote proves the hardware signed your description of the workload; proving that this exact
  code ran needs a measured launcher (for instance the Confidential Containers project with
  a key broker), which is outside this tool;
* anything about the VM's own software stack beyond what the TDX measurements (`MRTD`,
  `RTMR0..3`) say. The tool prints them; comparing them against a reference image is your
  policy, not the tool's;
* anything about a VM you did not attest yourself.

### The gap this tool does not close

Both proofs are bound to a description of the workload. Neither proof says that the
described image or model was what the GPU computed on. Closing that gap requires a measured
launch: a component inside the VM that measures the container image and the model before
starting them and extends a TDX runtime measurement (`RTMR`) with the result, so that the
quote itself carries the measurement. The Confidential Containers project (Kata + Trustee)
and similar stacks do this. `voltage-verify` stops one step earlier on purpose, and says so.

### About this particular bundle

Its challenge was drawn by the publisher's script, not by you, so freshness relative to you is not
shown here. For your own guarantee, rent a Confidential VM, draw the challenge on your machine and
run the attestation yourself: see `deploy-confidential-vm-api/` in this repository and
[`docs/ATTEST_ON_VM.md`](https://github.com/VoltageGPU/voltage-verify/blob/main/docs/ATTEST_ON_VM.md).